# Flash Attention (闪存注意力)

## 概述

Flash Attention是一种优化GPU内存访问模式的注意力计算方法，通过分块（tiling）技术显著减少高带宽内存（HBM）访问，同时保持数学上的完全等价性。

### 核心问题

**GPU内存层次**:
- **SRAM (片上内存)**: 快速但容量小 (~20MB)
- **HBM (高带宽内存)**: 容量大但慢 (~40GB)

**标准注意力的问题**:
```
1. 计算 QK^T → 写入HBM (n×n矩阵)
2. 读取 QK^T → 计算Softmax → 写入HBM
3. 读取注意力矩阵 → 乘以V → 写入HBM

问题: 大量的HBM读写操作，HBM带宽成为瓶颈
```

### Flash Attention的解决方案

**核心技术**:
1. **分块计算**: 将Q、K、V分成小块，一次只处理一个块
2. **在线Softmax**: 增量计算Softmax，避免存储完整注意力矩阵
3. **SRAM优先**: 尽可能在SRAM中完成计算

**优势**:
- 速度提升 2-4x
- 内存占用大幅降低
- 数学上完全等价

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

## 1. 标准注意力实现

In [ ]:
def softmax(x, axis=-1):
    """Softmax函数"""
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


class StandardAttention:
    """
    标准注意力实现
    
    内存访问模式:
    1. 读取Q、K、V
    2. 计算并存储 n×n 注意力矩阵
    3. 多次HBM读写
    """
    
    def __init__(self, embed_dim):
        self.embed_dim = embed_dim
        self.W_q = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_k = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_v = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
    
    def forward(self, x, return_stats=False):
        seq_len, embed_dim = x.shape
        
        # 投影
        Q = np.dot(x, self.W_q)
        K = np.dot(x, self.W_k)
        V = np.dot(x, self.W_v)
        
        # 计算注意力矩阵 (需要O(n²)内存)
        scores = np.dot(Q, K.T) / np.sqrt(embed_dim)
        attention = softmax(scores, axis=-1)
        
        # 输出
        output = np.dot(attention, V)
        
        if return_stats:
            stats = {
                'attention_matrix_size': seq_len * seq_len * 4,  # bytes
                'peak_memory': seq_len * seq_len * 4 + seq_len * embed_dim * 12
            }
            return output, attention, stats
        
        return output


# 测试
embed_dim = 32
seq_len = 64
x = np.random.randn(seq_len, embed_dim)

std_attn = StandardAttention(embed_dim)
output, attention, stats = std_attn.forward(x, return_stats=True)

print(f"标准注意力:")
print(f"  输入形状: {x.shape}")
print(f"  输出形状: {output.shape}")
print(f"  注意力矩阵形状: {attention.shape}")
print(f"  注意力矩阵大小: {stats['attention_matrix_size'] / 1024:.2f} KB")
print(f"  峰值内存: {stats['peak_memory'] / 1024:.2f} KB")

## 2. Flash Attention实现

核心技巧：分块计算 + 在线Softmax

In [ ]:
class FlashAttention:
    """
    Flash Attention实现
    
    核心技术:
    1. 分块计算 - 一次只处理小块数据
    2. 在线Softmax - 增量更新，无需存储完整矩阵
    3. SRAM优先 - 最大化片上内存利用
    """
    
    def __init__(self, embed_dim, block_size=32):
        self.embed_dim = embed_dim
        self.block_size = block_size
        
        self.W_q = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_k = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_v = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
    
    def _online_softmax_update(self, m_old, l_old, m_new, l_new):
        """
        在线Softmax更新
        
        当看到新的数据块时，需要更新之前的结果
        """
        m_global = np.maximum(m_old, m_new)
        
        correction_old = np.exp(m_old - m_global)
        correction_new = np.exp(m_new - m_global)
        
        l_global = correction_old * l_old + correction_new * l_new
        
        return m_global, l_global, correction_old
    
    def forward(self, x, return_stats=False):
        """
        Flash Attention前向传播
        
        使用分块避免存储完整注意力矩阵
        """
        seq_len, embed_dim = x.shape
        block_size = self.block_size
        
        # 投影
        Q = np.dot(x, self.W_q)
        K = np.dot(x, self.W_k)
        V = np.dot(x, self.W_v)
        
        # 初始化输出和统计量
        output = np.zeros((seq_len, embed_dim))
        m = np.full(seq_len, -np.inf)  # 最大值
        l = np.zeros(seq_len)  # 归一化因子
        
        num_blocks = (seq_len + block_size - 1) // block_size
        
        # 外循环: 遍历Q的块
        for i in range(num_blocks):
            q_start = i * block_size
            q_end = min((i + 1) * block_size, seq_len)
            Q_block = Q[q_start:q_end]
            
            # 当前块的累积
            O_block = np.zeros((q_end - q_start, embed_dim))
            m_block = np.full(q_end - q_start, -np.inf)
            l_block = np.zeros(q_end - q_start)
            
            # 内循环: 遍历K、V的块
            for j in range(num_blocks):
                kv_start = j * block_size
                kv_end = min((j + 1) * block_size, seq_len)
                K_block = K[kv_start:kv_end]
                V_block = V[kv_start:kv_end]
                
                # 计算当前块的注意力
                scores_block = np.dot(Q_block, K_block.T) / np.sqrt(embed_dim)
                
                # 当前块的统计量
                m_new = np.max(scores_block, axis=1)
                scores_shifted = scores_block - m_new[:, None]
                exp_scores = np.exp(scores_shifted)
                l_new = np.sum(exp_scores, axis=1)
                
                # 在线更新
                m_global, l_global, correction = self._online_softmax_update(
                    m_block, l_block, m_new, l_new
                )
                
                # 更新输出 (重新缩放旧值 + 新值)
                O_block = O_block * (correction * l_block / l_global)[:, None]
                attention_block = exp_scores / l_global[:, None]
                O_block += np.dot(attention_block, V_block)
                
                m_block = m_global
                l_block = l_global
            
            # 写回全局输出
            output[q_start:q_end] = O_block
        
        if return_stats:
            stats = {
                'max_block_size': block_size * block_size * 4,  # bytes
                'peak_memory': block_size * block_size * 4 + seq_len * embed_dim * 12
            }
            return output, stats
        
        return output


# 测试Flash Attention
flash_attn = FlashAttention(embed_dim, block_size=32)
output_flash, stats_flash = flash_attn.forward(x, return_stats=True)

print(f"\nFlash Attention:")
print(f"  输入形状: {x.shape}")
print(f"  输出形状: {output_flash.shape}")
print(f"  最大块大小: {stats_flash['max_block_size'] / 1024:.2f} KB")
print(f"  峰值内存: {stats_flash['peak_memory'] / 1024:.2f} KB")

# 验证正确性
diff = np.abs(output - output_flash)
print(f"\n正确性验证:")
print(f"  平均误差: {np.mean(diff):.2e}")
print(f"  最大误差: {np.max(diff):.2e}")
print(f"  结果一致: {np.allclose(output, output_flash, rtol=1e-5)}")

## 3. 性能对比

In [ ]:
def benchmark_comparison(seq_lengths, embed_dim=64, block_size=64):
    """
    对比不同序列长度下的性能
    """
    results = {
        'seq_lengths': seq_lengths,
        'std_times': [],
        'flash_times': [],
        'std_memory': [],
        'flash_memory': []
    }
    
    for seq_len in seq_lengths:
        x = np.random.randn(seq_len, embed_dim)
        
        # 标准注意力
        std_attn = StandardAttention(embed_dim)
        start = time.time()
        _, _, stats_std = std_attn.forward(x, return_stats=True)
        std_time = time.time() - start
        
        # Flash Attention
        flash_attn = FlashAttention(embed_dim, block_size=block_size)
        start = time.time()
        _, stats_flash = flash_attn.forward(x, return_stats=True)
        flash_time = time.time() - start
        
        results['std_times'].append(std_time * 1000)
        results['flash_times'].append(flash_time * 1000)
        results['std_memory'].append(stats_std['attention_matrix_size'] / (1024*1024))
        results['flash_memory'].append(stats_flash['max_block_size'] / (1024*1024))
    
    return results


# 运行基准测试
seq_lengths = [64, 128, 256, 512, 1024]
results = benchmark_comparison(seq_lengths, embed_dim=64, block_size=64)

# 打印结果
print("性能对比:")
print("-" * 80)
print(f"{'序列长度':<12} {'标准(ms)':<15} {'Flash(ms)':<15} {'加速比':<12} {'内存节省':<12}")
print("-" * 80)

for i, seq_len in enumerate(seq_lengths):
    std_time = results['std_times'][i]
    flash_time = results['flash_times'][i]
    speedup = std_time / flash_time
    memory_saving = (results['std_memory'][i] - results['flash_memory'][i]) / results['std_memory'][i]
    
    print(f"{seq_len:<12} {std_time:<15.4f} {flash_time:<15.4f} {speedup:<12.2f}x {memory_saving:<12.1%}")

print("-" * 80)

In [ ]:
# 绘制性能对比图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 时间对比
ax1 = axes[0]
ax1.plot(results['seq_lengths'], results['std_times'], 'o-', label='标准注意力', linewidth=2, markersize=8)
ax1.plot(results['seq_lengths'], results['flash_times'], 's-', label='Flash Attention', linewidth=2, markersize=8)
ax1.set_xlabel('序列长度', fontsize=12)
ax1.set_ylabel('时间 (ms)', fontsize=12)
ax1.set_title('计算时间对比', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# 内存对比
ax2 = axes[1]
ax2.plot(results['seq_lengths'], results['std_memory'], 'o-', label='标准注意力', linewidth=2, markersize=8)
ax2.plot(results['seq_lengths'], results['flash_memory'], 's-', label='Flash Attention', linewidth=2, markersize=8)
ax2.set_xlabel('序列长度', fontsize=12)
ax2.set_ylabel('内存占用 (MB)', fontsize=12)
ax2.set_title('内存占用对比', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

## 4. 块大小的影响

In [ ]:
# 测试不同块大小
seq_len = 512
embed_dim = 64
x = np.random.randn(seq_len, embed_dim)

block_sizes = [16, 32, 64, 128, 256]
times = []
memories = []

for block_size in block_sizes:
    flash_attn = FlashAttention(embed_dim, block_size=block_size)
    
    start = time.time()
    output, stats = flash_attn.forward(x, return_stats=True)
    elapsed = time.time() - start
    
    times.append(elapsed * 1000)
    memories.append(stats['max_block_size'] / 1024)  # KB

# 绘制
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(block_sizes, times, 'o-', linewidth=2, markersize=8, color='steelblue')
ax1.set_xlabel('块大小', fontsize=12)
ax1.set_ylabel('时间 (ms)', fontsize=12)
ax1.set_title('块大小 vs 计算时间', fontsize=14)
ax1.grid(True, alpha=0.3)

ax2.plot(block_sizes, memories, 's-', linewidth=2, markersize=8, color='coral')
ax2.set_xlabel('块大小', fontsize=12)
ax2.set_ylabel('块内存 (KB)', fontsize=12)
ax2.set_title('块大小 vs 内存占用', fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n块大小影响:")
print("-" * 50)
for bs, t, m in zip(block_sizes, times, memories):
    print(f"块大小={bs:<4}  时间={t:>8.4f}ms  内存={m:>8.2f}KB")

## 5. 内存访问模式可视化

In [ ]:
# 可视化内存访问模式
seq_len = 64
block_size = 16

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# 标准注意力: 需要完整的n×n矩阵
full_attention = np.ones((seq_len, seq_len))
im1 = ax1.imshow(full_attention, cmap='Reds', aspect='auto')
ax1.set_title('标准注意力内存访问\n需要完整的n×n矩阵', fontsize=13, fontweight='bold')
ax1.set_xlabel('Key位置', fontsize=11)
ax1.set_ylabel('Query位置', fontsize=11)
plt.colorbar(im1, ax=ax1)

# Flash Attention: 分块访问
blocked_attention = np.zeros((seq_len, seq_len))
num_blocks = seq_len // block_size

# 标记当前处理的块
for i in range(num_blocks):
    for j in range(num_blocks):
        q_start, q_end = i * block_size, (i + 1) * block_size
        k_start, k_end = j * block_size, (j + 1) * block_size
        blocked_attention[q_start:q_end, k_start:k_end] = (i * num_blocks + j + 1) / (num_blocks * num_blocks)

im2 = ax2.imshow(blocked_attention, cmap='Blues', aspect='auto')
ax2.set_title(f'Flash Attention内存访问\n分块处理 (块大小={block_size})', fontsize=13, fontweight='bold')
ax2.set_xlabel('Key位置', fontsize=11)
ax2.set_ylabel('Query位置', fontsize=11)
plt.colorbar(im2, ax=ax2)

# 添加网格线
for i in range(0, seq_len, block_size):
    ax2.axhline(y=i, color='white', linewidth=1)
    ax2.axvline(x=i, color='white', linewidth=1)

plt.tight_layout()
plt.show()

print(f"\n内存访问对比:")
print(f"标准注意力: 需要存储 {seq_len}×{seq_len} = {seq_len*seq_len} 个元素")
print(f"Flash Attention: 每次只需 {block_size}×{block_size} = {block_size*block_size} 个元素")
print(f"内存节省: {(1 - block_size*block_size/(seq_len*seq_len))*100:.1f}%")

## 6. 在线Softmax原理演示

In [ ]:
# 演示在线Softmax如何工作
def naive_softmax(x):
    """标准Softmax (需要看到所有数据)"""
    m = np.max(x)
    exp_x = np.exp(x - m)
    return exp_x / np.sum(exp_x)


def online_softmax(x, chunk_size=2):
    """在线Softmax (逐块处理)"""
    n = len(x)
    m_global = -np.inf
    l_global = 0
    result = np.zeros(n)
    
    num_chunks = (n + chunk_size - 1) // chunk_size
    
    for i in range(num_chunks):
        start = i * chunk_size
        end = min((i + 1) * chunk_size, n)
        x_chunk = x[start:end]
        
        # 当前块的统计量
        m_new = np.max(x_chunk)
        exp_chunk = np.exp(x_chunk - m_new)
        l_new = np.sum(exp_chunk)
        
        # 更新全局统计量
        m_old = m_global
        m_global = max(m_old, m_new)
        
        # 修正之前的结果
        correction_old = np.exp(m_old - m_global)
        correction_new = np.exp(m_new - m_global)
        
        result[:start] *= correction_old
        result[start:end] = exp_chunk * correction_new
        
        l_global = correction_old * l_global + correction_new * l_new
    
    # 最终归一化
    result /= l_global
    return result


# 测试
x_test = np.random.randn(8)

naive_result = naive_softmax(x_test)
online_result = online_softmax(x_test, chunk_size=2)

print("在线Softmax验证:")
print(f"输入: {x_test}")
print(f"\n标准Softmax: {naive_result}")
print(f"在线Softmax:  {online_result}")
print(f"\n误差: {np.max(np.abs(naive_result - online_result)):.2e}")
print(f"结果一致: {np.allclose(naive_result, online_result)}")

# 可视化
x_range = np.arange(len(x_test))
width = 0.35

plt.figure(figsize=(10, 6))
plt.bar(x_range - width/2, naive_result, width, label='标准Softmax', alpha=0.8)
plt.bar(x_range + width/2, online_result, width, label='在线Softmax', alpha=0.8)
plt.xlabel('位置', fontsize=12)
plt.ylabel('Softmax输出', fontsize=12)
plt.title('标准Softmax vs 在线Softmax', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 7. 总结

### Flash Attention的核心优势

1. **速度提升**: 2-4x加速
   - 减少HBM访问次数
   - 充分利用SRAM

2. **内存节省**: 不需要O(n²)的注意力矩阵
   - 标准: 需要存储n×n矩阵
   - Flash: 只需要block_size×block_size

3. **数学等价**: 完全相同的输出
   - 不是近似方法
   - 在线Softmax保证正确性

4. **可扩展性**: 支持更长序列
   - 内存不随序列长度平方增长
   - 适合长文档、长视频

### 核心技术

- **分块计算 (Tiling)**: 降低内存峰值
- **在线Softmax**: 增量更新，无需存储完整矩阵
- **重计算**: 前向不保存，反向重新计算
- **IO优化**: SRAM优先，最小化HBM访问

### 实际应用

- GPT-4
- Llama 2/3
- Claude
- 所有现代LLM训练和推理

### 块大小选择

- 太小: 计算开销大
- 太大: 超出SRAM容量
- 典型值: 64-128
- 需要根据硬件调优